# Jev SOC analysis

Recompute the saved scores and token usage. Start Jupyter in `rigorous/` after running setup. These cells read local traces and make no API calls.

In [1]:
import json
from pathlib import Path
from collections import Counter
root = Path.cwd()
assert (root/'evidence/frozen-plan.json').exists()
primary = [json.loads(line) for line in (root/'results/calls.jsonl').read_text().splitlines()]
rows = {row['id']: row for row in primary}
cases = json.loads((root/'evidence/malware-cases.json').read_text())
print('Primary calls:', len(primary))
print('Unique reports:', len({c['report_id'] for c in cases}))
print(Counter(c['split'] for c in cases))

Primary calls: 2695
Unique reports: 175
Counter({'test': 454, 'dev': 155})


In [2]:
official = json.loads((root/'vendor/meta-llama_PurpleLlama/CybersecurityBenchmarks/datasets/crwd_meta/malware_analysis/questions.json').read_text())
for arm in ['retrieved', 'prefix', 'none']:
    n = correct = 0
    for i, question in enumerate(official):
        if cases[i]['split'] != 'test': continue
        answers = rows[f'M{i:04d}/{arm}']['response']['answers']
        selected = {key[7:] for key, value in answers.items() if value['noul'] >= .5}
        correct += selected == set(question['correct_options'])
        n += 1
    print(arm, correct, '/', n, '=', round(100*correct/n, 2), '%')

retrieved 120 / 454 = 26.43 %
prefix 75 / 454 = 16.52 %
none 36 / 454 = 7.93 %


In [3]:
policy = json.loads((root/'evidence/policy-cases.json').read_text())
for field in ['established', 'recommend', 'execute', 'completed']:
    false_positive = negatives = false_negative = positives = 0
    for case in policy:
        gold = case['gold'][field]
        selected = rows[case['id']+'/policy']['response']['answers'][field]['noul'] >= .5
        negatives += not gold
        positives += gold
        false_positive += selected and not gold
        false_negative += not selected and gold
    print(field, 'FP', false_positive, '/', negatives, 'FN', false_negative, '/', positives)

established FP 0 / 32 FN 0 / 224
recommend FP 8 / 48 FN 0 / 208
execute FP 42 / 144 FN 16 / 112
completed FP 0 / 64 FN 3 / 192


In [4]:
all_rows = primary + [json.loads(line) for filename in ['followup-calls.jsonl','repeat-calls.jsonl'] for line in (root/'results'/filename).read_text().splitlines()]
tokens = sum(row['response']['usage']['input_tokens'] for row in all_rows)
print('All calls:', len(all_rows), 'input tokens:', tokens, 'estimated USD:', tokens*.042/1_000_000)
assert len({row['id'] for row in all_rows}) == len(all_rows)

All calls: 3364 input tokens: 25102078 estimated USD: 1.0542872760000002


## Figures

![Primary results](figures/pipeline-results.png)

![Calibration](figures/calibration.png)

See `REPORT.md`, `CASE_AUDIT.md`, `VALIDATION.md` and `results/independent-validation.json`. Report-cluster and template-cluster bootstrap calculations are in `analyze.py`. The compact adapter and stability controls are explicitly exploratory.